# Trinetra Air Quality ML Pipeline
### Pollution Source Classification & Multi-Variable AQI Forecasting

This notebook trains and evaluates two machine learning models on the campus sensor dataset (`Somaiya_Campus_All_17_Locations_Combined.csv`):

1. **Random Forest Classifier**: Pollution level classification (`Dust_Category`) without target leakage or overfitting.
2. **Multi-Output LSTM Neural Network**: Joint 1-step ahead forecasting of **Dust Density (ug/m3), MQ135 Gas (PPM), Temperature (C), and Humidity (%)** using per-location sequence windowing.

---

## 1. Environment Setup & Dependencies

In [ ]:
import numpy as np
import pandas as pd
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print("Environment initialized successfully.")

ModuleNotFoundError: No module named 'matplotlib'

## 2. Data Loading & Feature Engineering

The dataset contains time-series sensor telemetry across 17 campus locations.

In [ ]:
# Path resolution: check local directory first
possible_paths = [
    'Somaiya_Campus_All_17_Locations_Combined.csv',
    '../Somaiya_Campus_All_17_Locations_Combined.csv',
    'data/processed/air_quality_data.csv'
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path:
    df = pd.read_csv(data_path)
    print(f"Loaded dataset from '{data_path}': {len(df)} rows, {len(df.columns)} columns")
else:
    print("Dataset file not found - generating synthetic dataset for demonstration...")
    n_samples = 3000
    df = pd.DataFrame({
        'Timestamp': pd.date_range('2026-08-14', periods=n_samples, freq='1min'),
        'Location': np.random.choice(['Location_A', 'Location_B', 'Location_C'], n_samples),
        'Air_Quality_MQ135': np.random.uniform(150, 400, n_samples),
        'Dust_Density_ug_m3': np.random.uniform(10, 150, n_samples),
        'Temperature_C': np.random.uniform(20, 38, n_samples),
        'Humidity_Pct': np.random.uniform(40, 90, n_samples),
        'Dew_Point_C': np.random.uniform(15, 25, n_samples),
        'VPD_kPa': np.random.uniform(0.3, 1.2, n_samples),
    })
    df['Dust_Category'] = pd.cut(df['Dust_Density_ug_m3'], bins=[0, 30, 60, 90, 200], labels=['Good', 'Moderate', 'Poor', 'Unhealthy'])

# Clean & sort chronologically per location
if 'Timestamp' in df.columns:
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df = df.sort_values(['Location', 'Timestamp']).reset_index(drop=True)

# Feature Engineering: Cyclical hour encoding to capture diurnal cycles
if 'Hour_Of_Day' in df.columns:
    hours = df['Hour_Of_Day']
elif 'Timestamp' in df.columns:
    hours = df['Timestamp'].dt.hour
else:
    hours = np.zeros(len(df))

df['sin_hour'] = np.sin(2 * np.pi * hours / 24.0)
df['cos_hour'] = np.cos(2 * np.pi * hours / 24.0)

# Impute any NaNs
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].ffill().bfill()

print(f"Dataset ready. Total locations: {df['Location'].nunique()}")
print(df[['Location', 'Dust_Density_ug_m3', 'Air_Quality_MQ135', 'Dust_Category', 'sin_hour', 'cos_hour']].head())

## 3. Model 1 — Random Forest Classifier (Pollution Level Classification)

### Overfitting Prevention Strategy:
1. **Eliminate Target Leakage**: Remove direct derivative metrics (`Dust_Sub_AQI`, `MQ135_Status`, etc.) from features.
2. **Per-Location Chronological Split**: Sequential 80% train / 10% val / 10% test split per location to prevent temporal auto-correlation leakage.
3. **Regularized Hyperparameters**: Restrict max tree depth (`max_depth=8`) and minimum leaf size (`min_samples_leaf=10`).

In [ ]:
# Features (raw sensors & physical metrics only -- NO calculated AQI/Category targets)
rf_features = ['Air_Quality_MQ135', 'Dust_Density_ug_m3', 'Temperature_C', 'Humidity_Pct', 'Dew_Point_C', 'VPD_kPa', 'sin_hour', 'cos_hour']
# Ensure features exist in df
rf_features = [col for col in rf_features if col in df.columns]
target_col = 'Dust_Category'

le = LabelEncoder()
y_rf = le.fit_transform(df[target_col].astype(str))
class_names = list(le.classes_)

# Perform chronological split per location
train_indices, val_indices, test_indices = [], [], []
for loc in df['Location'].unique():
    loc_idx = df[df['Location'] == loc].index.values
    n = len(loc_idx)
    t_end = int(n * 0.8)
    v_end = int(n * 0.9)
    train_indices.extend(loc_idx[:t_end])
    val_indices.extend(loc_idx[t_end:v_end])
    test_indices.extend(loc_idx[v_end:])

X_train_rf = df.iloc[train_indices][rf_features]
y_train_rf = y_rf[train_indices]
X_val_rf = df.iloc[val_indices][rf_features]
y_val_rf = y_rf[val_indices]
X_test_rf = df.iloc[test_indices][rf_features]
y_test_rf = y_rf[test_indices]

# Standard scaling on train split
scaler_rf = StandardScaler()
X_train_rf_s = scaler_rf.fit_transform(X_train_rf)
X_val_rf_s = scaler_rf.transform(X_val_rf)
X_test_rf_s = scaler_rf.transform(X_test_rf)

# Regularized Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=10,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_rf_s, y_train_rf)

# Evaluation
y_train_pred_rf = rf_model.predict(X_train_rf_s)
y_test_pred_rf = rf_model.predict(X_test_rf_s)

acc_train = accuracy_score(y_train_rf, y_train_pred_rf)
acc_test = accuracy_score(y_test_rf, y_test_pred_rf)
f1_train = precision_recall_fscore_support(y_train_rf, y_train_pred_rf, average='weighted')[2]
f1_test = precision_recall_fscore_support(y_test_rf, y_test_pred_rf, average='weighted')[2]

print("=== Random Forest Results ===")
print(f"Train Accuracy: {acc_train*100:.2f}%  |  Test Accuracy: {acc_test*100:.2f}%")
print(f"Train F1-Score: {f1_train*100:.2f}%  |  Test F1-Score: {f1_test*100:.2f}%")
print(f"Accuracy Gap (Train - Test): {(acc_train - acc_test)*100:.2f}%")

if (acc_train - acc_test) < 0.08:
    print("Verdict: WELL-GENERALIZED (No overfitting detected)")
else:
    print("Verdict: OVERFITTING DETECTED (Adjust depth or min_samples_leaf)")

In [ ]:
# Visualizations: Confusion Matrix & Feature Importances
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm_test = confusion_matrix(y_test_rf, y_test_pred_rf, normalize='true') * 100
sns.heatmap(cm_test, annot=True, fmt='.1f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('RF Test Confusion Matrix (%)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Feature Importance
importances = pd.Series(rf_model.feature_importances_, index=rf_features).sort_values(ascending=True)
importances.plot(kind='barh', color='#2b5c8f', ax=axes[1])
axes[1].set_title('Random Forest Feature Importance')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

## 4. Model 2 — Multi-Output LSTM Neural Network (AQI & Telemetry Forecasting)

### Underfitting Prevention Strategy:
1. **Per-Location Window Construction**: Build rolling sequences ($lookback = 15$) strictly within each location to eliminate cross-location sequence bleeding.
2. **MinMax Feature Scaling**: Fit scaler on train split sequences for smooth gradient propagation.
3. **Sufficient Network Capacity**: 2-layer LSTM ($64 \rightarrow 32$ units) with Dense hidden layer and moderate dropout ($0.1$).

In [ ]:
lstm_input_cols = ['Dust_Density_ug_m3', 'Air_Quality_MQ135', 'Temperature_C', 'Humidity_Pct', 'sin_hour', 'cos_hour']
lstm_input_cols = [c for c in lstm_input_cols if c in df.columns]
forecast_targets = ['Dust_Density_ug_m3', 'Air_Quality_MQ135', 'Temperature_C', 'Humidity_Pct']
forecast_targets = [c for c in forecast_targets if c in df.columns]

seq_len = 15

# Split locations into per-location chunks before creating sequences
train_chunks, val_chunks, test_chunks = [], [], []
for loc in df['Location'].unique():
    loc_df = df[df['Location'] == loc].reset_index(drop=True)
    n = len(loc_df)
    t_end = int(n * 0.8)
    v_end = int(n * 0.9)
    train_chunks.append(loc_df.iloc[:t_end])
    val_chunks.append(loc_df.iloc[t_end:v_end])
    test_chunks.append(loc_df.iloc[v_end:])

# Fit scalers on train split only
scaler_lstm_X = MinMaxScaler()
scaler_lstm_y = MinMaxScaler()

train_combined_df = pd.concat(train_chunks, ignore_index=True)
scaler_lstm_X.fit(train_combined_df[lstm_input_cols])
scaler_lstm_y.fit(train_combined_df[forecast_targets])

def create_per_location_sequences(chunks):
    X_list, y_list = [], []
    for chunk in chunks:
        if len(chunk) <= seq_len:
            continue
        x_scaled = scaler_lstm_X.transform(chunk[lstm_input_cols])
        y_scaled = scaler_lstm_y.transform(chunk[forecast_targets])
        for i in range(len(chunk) - seq_len):
            X_list.append(x_scaled[i:i+seq_len])
            y_list.append(y_scaled[i+seq_len])
    return np.array(X_list), np.array(y_list)

X_train_l, y_train_l = create_per_location_sequences(train_chunks)
X_val_l, y_val_l = create_per_location_sequences(val_chunks)
X_test_l, y_test_l = create_per_location_sequences(test_chunks)

print(f"Sequence Pool Shapes:")
print(f"  Train: X={X_train_l.shape}, y={y_train_l.shape}")
print(f"  Val:   X={X_val_l.shape}, y={y_val_l.shape}")
print(f"  Test:  X={X_test_l.shape}, y={y_test_l.shape}")

In [ ]:
# Build Multi-Output LSTM Architecture
n_features = len(lstm_input_cols)
n_targets = len(forecast_targets)

lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(seq_len, n_features)),
    Dropout(0.1),
    LSTM(32),
    Dropout(0.1),
    Dense(16, activation='relu'),
    Dense(n_targets)
])

lstm_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
lstm_model.summary()

# Train with EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = lstm_model.fit(
    X_train_l, y_train_l,
    validation_data=(X_val_l, y_val_l),
    epochs=40,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# LSTM Evaluation: Inverse Transform to Original Physical Units
y_train_pred_s = lstm_model.predict(X_train_l)
y_test_pred_s = lstm_model.predict(X_test_l)

y_train_act = scaler_lstm_y.inverse_transform(y_train_l)
y_train_pred = scaler_lstm_y.inverse_transform(y_train_pred_s)

y_test_act = scaler_lstm_y.inverse_transform(y_test_l)
y_test_pred = scaler_lstm_y.inverse_transform(y_test_pred_s)

metrics_list = []
for i, target in enumerate(forecast_targets):
    r2_tr = r2_score(y_train_act[:, i], y_train_pred[:, i])
    r2_te = r2_score(y_test_act[:, i], y_test_pred[:, i])
    mae_te = mean_absolute_error(y_test_act[:, i], y_test_pred[:, i])
    rmse_te = np.sqrt(mean_squared_error(y_test_act[:, i], y_test_pred[:, i]))
    metrics_list.append({
        'Target Variable': target,
        'Train R2': round(r2_tr, 4),
        'Test R2': round(r2_te, 4),
        'Test MAE': round(mae_te, 4),
        'Test RMSE': round(rmse_te, 4)
    })

metrics_df = pd.DataFrame(metrics_list)
print("=== Multi-Output LSTM Forecasting Performance ===")
print(metrics_df.to_string(index=False))

overall_r2 = r2_score(y_test_act, y_test_pred)
print(f"\nOverall Test R^2 Score across all targets: {overall_r2:.4f}")

In [ ]:
# Visualizations: Loss Curve & Forecast vs Actual Plots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, target in enumerate(forecast_targets[:4]):
    # Plot first 150 test samples for visual clarity
    n_plot = min(150, len(y_test_act))
    axes[i].plot(y_test_act[:n_plot, i], label='Actual', color='black', alpha=0.7, linewidth=1.5)
    axes[i].plot(y_test_pred[:n_plot, i], label='LSTM Forecast', color='#d95f02', linestyle='--', linewidth=1.5)
    axes[i].set_title(f'1-Step Forecast: {target}')
    axes[i].set_xlabel('Sample Step (Minutes)')
    axes[i].set_ylabel('Value')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Multi-Output LSTM Actual vs Forecast Predictions (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Model Serialization & Export

In [ ]:
os.makedirs('models', exist_ok=True)

# Export Random Forest model and scaler
joblib.dump(rf_model, 'models/rf_classifier.pkl')
joblib.dump(scaler_rf, 'models/rf_scaler.pkl')
joblib.dump(le, 'models/label_encoder.pkl')

# Export LSTM model and scalers
lstm_model.save('models/lstm_forecaster.h5')
joblib.dump(scaler_lstm_X, 'models/scaler_lstm_X.pkl')
joblib.dump(scaler_lstm_y, 'models/scaler_lstm_y.pkl')

print("All trained models and scalers saved to 'models/' directory successfully!")